> **Tip**: Welcome to the Investigate a Dataset project! You will find tips in quoted sections like this to help organize your approach to your investigation. Once you complete this project, remove these **Tip** sections from your report before submission. First things first, you might want to double-click this Markdown cell and change the title so that it reflects your dataset and investigation.

# Project: Investigate a Dataset - Medical Appointment No Shows

## Table of Contents
<ul>
<li><a href="#intro">Introduction</a></li>
<li><a href="#wrangling">Data Wrangling</a></li>
<li><a href="#eda">Exploratory Data Analysis</a></li>
<li><a href="#conclusions">Conclusions</a></li>
</ul>

<a id='intro'></a>
## Introduction

<a id='dataset'></a>
### Dataset Description 

We will be analyzing a dataset containing information about medical appointments made between Nov 10, 2015 and June 8, 2016 in the city of Vitória, Brazil, with the aim of investigating factors that may correlate with missed appointments.

The dataset was sourced from [Kaggle: Medical Appointment No Shows](https://www.kaggle.com/datasets/joniarroba/noshowappointments). It contains data on 100K+ medical appointments with information about each appointment and the patient. For this analysis, some columns have been renamed for clarity.

| column name | description | original name |
|---|---|---|
| `PatientId` | unique identifier for each patient |`PatientId`|
| `AppointmentId` | unique identifier for each appointment |`AppointmentID`|
| `Gender` | Male or Female |`Gender`|
| `ScheduledDate` | date of the appointment - [reference](https://www.kaggle.com/datasets/joniarroba/noshowappointments) |`ScheduledDay`|
| `AppointmentCreatedDate` | date the appointment was created - [reference](https://www.kaggle.com/datasets/joniarroba/noshowappointments) |`AppointmentDay`|
| `Age` | age of the patient |`Age`|
| `Neighborhood` | neighborhood of Vitória, Brazil where the appointment takes place |`Neighbourhood`|
| `FinancialAssistance` | patient enrolled in [Bolsa Família](https://en.wikipedia.org/wiki/Bolsa_Fam%C3%ADlia) financial assistance program |`Scholarship`|
| `Hypertension` | patient diagnosed with hypertension |`Hipertension`|
| `Diabetes` | patient diagnosed with diabetes |`Diabetes`|
| `Alcoholism` | patient diagnosed with alcoholism |`Alcoholism`|
| `NumHandicaps` | number of handicaps - [reference](https://www.kaggle.com/datasets/joniarroba/noshowappointments/discussion/29699#229356) |`Handcap`|
| `SMS_received` | 1 or more text messages were sent to the patient |`SMS_received`|
| `NoShow` | the patient missed the appointment |`No-show`|


### Question(s) for Analysis
**Question 1:** Are there any factors specific to a patient that correlate to a missed appointment

**Question 2:** Are there any factors related to the appointment itself that correlate to a missed appointment?

<a id='wrangling'></a>
## Data Wrangling

### Environment Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

### General Properties
- 110527 records found
- no empty fields found

In [2]:
df = pd.read_csv(
    "./Database_No_show_appointments/noshowappointments-kagglev2-may-2016.csv"
)

df = df.convert_dtypes()

print(f"rows: {df.shape[0]}")
print(f"any na?: {df.isna().sum().any()}")
print(f"any null?: {df.isnull().sum().any()}\n")

print(df.info())

rows: 110527
any na?: False
any null?: False

<class 'pandas.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  Float64
 1   AppointmentID   110527 non-null  Int64  
 2   Gender          110527 non-null  string 
 3   ScheduledDay    110527 non-null  string 
 4   AppointmentDay  110527 non-null  string 
 5   Age             110527 non-null  Int64  
 6   Neighbourhood   110527 non-null  string 
 7   Scholarship     110527 non-null  Int64  
 8   Hipertension    110527 non-null  Int64  
 9   Diabetes        110527 non-null  Int64  
 10  Alcoholism      110527 non-null  Int64  
 11  Handcap         110527 non-null  Int64  
 12  SMS_received    110527 non-null  Int64  
 13  No-show         110527 non-null  string 
dtypes: Float64(1), Int64(8), string(5)
memory usage: 12.8 MB
None


### Data Cleaning
#### Column naming
Renaming columns for clarity - [see Dataset Description](#dataset) for more info
- `AppointmentID` --> `AppointmentId` for consistency
- `ScheduledDay` --> `ScheduledDate` to better reflect data type
- `AppointmentDay` --> `AppointmentCreatedDate` for clarity and to reflect data type
- `Neighbourhood` --> `Neigborhood` (US English spelling)
- `Scholarship` --> `FinancialAssistance` for clarity
- `Hipertension` --> `Hypertension` (US English spelling)
- `Handcap` --> `NumHandicaps` for clarity
- `No-show` --> `NoShow` for consitency and Python compatibility

In [3]:
df.rename(
    columns={
        "AppointmentID": "AppointmentId",
        "ScheduledDay": "ScheduledDate",
        "AppointmentDay": "AppointmentCreatedDate",
        "Neighbourhood": "Neighborhood",
        "Scholarship": "FinancialAssistance",
        "Hipertension": "Hypertension",
        "Handcap": "NumHandicaps",
        "No-show": "NoShow"
    },
    inplace=True,
)

#### AppointmentId
- no duplicate `AppointmentId`s found
- `AppointmentId` will be used as the DataFrame index

In [5]:
print(f"duplicate AppointmentIds: {df['AppointmentId'].duplicated().sum()}\n")
df.set_index("AppointmentId", inplace=True)

duplicate AppointmentIds: 0



#### ScheduledDate and AppointmentCreatedDate
- `ScheduledDate` and `AppointmentCreatedDate` converted to `datetime` datatype
- timestamps are GMT

In [6]:
df["ScheduledDate"] = pd.to_datetime(df["ScheduledDate"])
df["AppointmentCreatedDate"] = pd.to_datetime(df["AppointmentCreatedDate"])

#### Age
- Droped rows with invalid Age values `Age < 0`
- `Age == 0` assumed to be an infant
- Right-sized data type to cover realistic range of ages

In [7]:
print("Age range:")
print(f'{df["Age"].nunique()} unique (min:{df["Age"].min()}, max:{df["Age"].max()})')
df = df[df["Age"] >= 0]
print(f'{df["Age"].nunique()} unique (min:{df["Age"].min()}, max:{df["Age"].max()})')
df["Age"] = df["Age"].astype("Int8")

Age range:
104 unique (min:-1, max:115)
103 unique (min:0, max:115)


#### Neighborhood

- 81 unique neighborhoods
- converted to categorical datatype

In [8]:
print(f"neighborhood count: {df['Neighborhood'].nunique()}")
neighborhood_categories = df["Neighborhood"].unique()

neighborhood_category = pd.api.types.CategoricalDtype(
    ordered=False, categories=neighborhood_categories)

df["Neighborhood"] = df["Neighborhood"].astype(neighborhood_category)

neighborhood count: 81


#### Gender
- only 2 values `["F", "M"]` found in data
- `Gender` converted to categorical datatype `["female", "male"]`

In [9]:
print(df["Gender"].value_counts())

df["Gender"] = df["Gender"].map({"F": "female", "M": "male"})
gender_categories = df["Gender"].unique()

gender_category = pd.api.types.CategoricalDtype(
    ordered=False, categories=gender_categories)

df["Gender"] = df["Gender"].astype(gender_category)

print(df["Gender"].value_counts())

Gender
F    71839
M    38687
Name: count, dtype: Int64
Gender
female    71839
male      38687
Name: count, dtype: int64


##### Encoded Booleans
Converted binary columns (`[0,1]`, `["Yes","No"]`) to boolean
- `FinancialAssistance`,
- `Hypertension`,
- `Diabetes`,
- `Alcoholism`,
- `SMS_received`,
- `NoShow`

In [10]:
binaryColumns = [
    "FinancialAssistance",
    "Hypertension",
    "Diabetes",
    "Alcoholism",
    "SMS_received",
    "NoShow",
]

for colName in binaryColumns:
    print(f"{df[colName].value_counts()}\n")

df["FinancialAssistance"] = df["FinancialAssistance"].astype("boolean")
df["Hypertension"] = df["Hypertension"].astype("boolean")
df["Diabetes"] = df["Diabetes"].astype("boolean")
df["Alcoholism"] = df["Alcoholism"].astype("boolean")
df["SMS_received"] = df["SMS_received"].astype("boolean")
df["NoShow"] = df["NoShow"].map({"Yes": True, "No": False}).astype("boolean")

FinancialAssistance
0    99665
1    10861
Name: count, dtype: Int64

Hypertension
0    88725
1    21801
Name: count, dtype: Int64

Diabetes
0    102583
1      7943
Name: count, dtype: Int64

Alcoholism
0    107166
1      3360
Name: count, dtype: Int64

SMS_received
0    75044
1    35482
Name: count, dtype: Int64

NoShow
No     88207
Yes    22319
Name: count, dtype: Int64



#### numHandcap
- Right-sized datatype to fit value range

In [12]:
print(df["NumHandicaps"].value_counts())
df["NumHandicaps"] = df["NumHandicaps"].astype("Int8")

NumHandicaps
0    108285
1      2042
2       183
3        13
4         3
Name: count, dtype: Int64


#### PatientId

There are many appointments with recurring `PatientId` values.

Within each duplicate `PatientId`, there are no observed differences across:
- `Gender`
- `Hypertension`
- `Diabetes`
- `Alcoholism`
- `numHandicaps`
- `Neighborhood`
- `FinancialAssistance`

The dataset will be examined in terms of individual appointments and individual patients, therefore rows with duplicate `PatientId`s will not be dropped

In [14]:
print(f"unique PatientIds: {df['PatientId'].nunique()}")
print(f"duplicated PatientIds: {df['PatientId'].duplicated().sum()}")
print(f"duplicate PatientIds incl other factors: {df.duplicated(subset=[
        "PatientId",
        "Gender",
        "Hypertension",
        "Diabetes",
        "Alcoholism",
        "NumHandicaps",
        "Neighborhood",
        "FinancialAssistance"
        ]).sum()}")

unique PatientIds: 62298
duplicated PatientIds: 48228
duplicate PatientIds incl other factors: 48228


#### Data Cleanup results

In [15]:
df.info()

<class 'pandas.DataFrame'>
Index: 110526 entries, 5642903 to 5629448
Data columns (total 13 columns):
 #   Column                  Non-Null Count   Dtype              
---  ------                  --------------   -----              
 0   PatientId               110526 non-null  Float64            
 1   Gender                  110526 non-null  category           
 2   ScheduledDate           110526 non-null  datetime64[us, UTC]
 3   AppointmentCreatedDate  110526 non-null  datetime64[us, UTC]
 4   Age                     110526 non-null  Int8               
 5   Neighborhood            110526 non-null  category           
 6   FinancialAssistance     110526 non-null  boolean            
 7   Hypertension            110526 non-null  boolean            
 8   Diabetes                110526 non-null  boolean            
 9   Alcoholism              110526 non-null  boolean            
 10  NumHandicaps            110526 non-null  Int8               
 11  SMS_received            110526 non-

In [16]:
df.head()

,PatientId,Gender,ScheduledDate,AppointmentCreatedDate,Age,Neighborhood,FinancialAssistance,Hypertension,Diabetes,Alcoholism,NumHandicaps,SMS_received,NoShow
AppointmentId,,,,,,,,,,,,,
5642903,29872499824296.0,female,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,False,True,False,False,0,False,False
5642503,558997776694438.0,male,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,False,False,False,0,False,False
5642549,4262962299951.0,female,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,False,False,False,False,0,False,False
5642828,867951213174.0,female,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,False,False,False,False,0,False,False
5642494,8841186448183.0,female,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,False,True,True,False,0,False,False


<a id='eda'></a>
## Exploratory Data Analysis

> **Tip**: Now that you've trimmed and cleaned your data, you're ready to move on to exploration. **Compute statistics** and **create visualizations** with the goal of addressing the research questions that you posed in the Introduction section. You should compute the relevant statistics throughout the analysis when an inference is made about the data. Note that at least two or more kinds of plots should be created as part of the exploration, and you must  compare and show trends in the varied visualizations. Remember to utilize the visualizations that the pandas library already has available.



> **Tip**: Investigate the stated question(s) from multiple angles. It is recommended that you be systematic with your approach. Look at one variable at a time, and then follow it up by looking at relationships between variables. You should explore at least three variables in relation to the primary question. This can be an exploratory relationship between three variables of interest, or looking at how two independent variables relate to a single dependent variable of interest. Lastly, you  should perform both single-variable (1d) and multiple-variable (2d) explorations.


### Research Question 1 (Replace this header name!)

In [ ]:
# Use this, and more code cells, to explore your data. Don't forget to add
#   Markdown cells to document your observations and findings.

### Research Question 2  (Replace this header name!)

In [ ]:
# Continue to explore the data to address your additional research
#   questions. Add more headers as needed if you have more questions to
#   investigate.

<a id='conclusions'></a>
## Conclusions

> **Tip**: Finally, summarize your findings and the results that have been performed in relation to the question(s) provided at the beginning of the analysis. Summarize the results accurately, and point out where additional research can be done or where additional information could be useful.

> **Tip**: Make sure that you are clear with regards to the limitations of your exploration. You should have at least 1 limitation explained clearly. 

> **Tip**: If you haven't done any statistical tests, do not imply any statistical conclusions. And make sure you avoid implying causation from correlation!

> **Tip**: Once you are satisfied with your work here, check over your report to make sure that it is satisfies all the areas of the rubric (found on the project submission page at the end of the lesson). You should also probably remove all of the "Tips" like this one so that the presentation is as polished as possible.

## Submitting your Project 

> **Tip**: Before you submit your project, you need to create a .html or .pdf version of this notebook in the workspace here. To do that, run the code cell below. If it worked correctly, you should see output that starts with `NbConvertApp] Converting notebook`, and you should see the generated .html file in the workspace directory (click on the orange Jupyter icon in the upper left).

> **Tip**: Alternatively, you can download this report as .html via the **File** > **Download as** submenu, and then manually upload it into the workspace directory by clicking on the orange Jupyter icon in the upper left, then using the Upload button.

> **Tip**: Once you've done this, you can submit your project by clicking on the "Submit Project" button in the lower right here. This will create and submit a zip file with this .ipynb doc and the .html or .pdf version you created. Congratulations!

In [ ]:
# Running this cell will execute a bash command to convert this notebook to an .html file
!python -m nbconvert --to html Investigate_a_Dataset.ipynb